# 05_119_챗봇_연동확인
화면 오른쪽 아래 챗봇 창이 쓰는 `/api/chat` 동작 확인

- 먼저 별도 CMD에서 Flask 서버 실행 (`chatbot_exaone.py`가 연결된 상태)
- 이 노트북은 모델을 다시 불러오지 않고, 서버에 질문만 보냄

In [ ]:
import sys
import requests
import pandas as pd
from IPython.display import display

# chatbot_exaone.py가 있는 폴더를 파이썬 검색 경로에 추가
CHATBOT_DIR = r"C:\ai\source\09_Project\cheolhyeon_jo\html"
sys.path.insert(0, CHATBOT_DIR)
from chatbot_exaone import find_district, find_topics

BASE_URL = "http://127.0.0.1:5000"

## 1. 질문 분석 확인
질문 속 자치구 이름과 질문 종류(병상 / 이송 / 인구)를 찾는 규칙

In [ ]:
questions = ["강남 응급실 병상 알려줘", "마포구 병원 어디 있어?",
             "다음 달 이송 건수 예측은?", "송파구 생활인구 밀집", "안녕하세요"]

display(pd.DataFrame({"질문": questions,
                      "자치구": [find_district(q) for q in questions],
                      "질문 종류": [find_topics(q) for q in questions]}))

## 2. 모델 상태 확인
ready = EXAONE 답변 / loading = 불러오는 중 / error = 데이터 요약 답변으로 동작

In [ ]:
try:
    res = requests.get(BASE_URL + "/api/chat/status", timeout=10)
except requests.exceptions.ConnectionError:
    raise SystemExit(f"{BASE_URL} 서버가 꺼져 있습니다. CMD에서 Flask 서버를 먼저 실행하세요.")

if res.headers.get("Content-Type", "").startswith("application/json"):
    display(pd.DataFrame([res.json()]))
else:
    # json이 아니면 챗봇 주소가 서버에 연결되지 않은 상태
    display(pd.DataFrame([{"응답 코드": res.status_code,
                           "응답 형식": res.headers.get("Content-Type"),
                           "응답 앞부분": res.text[:150]}]))

## 3. 챗봇 질문 테스트
챗봇 창과 같은 방식으로 질문 + 선택 자치구를 보냄 (CPU에서는 한 질문에 수십 초 걸릴 수 있음)

In [ ]:
rows = []
for q in questions[:3]:
    res = requests.post(BASE_URL + "/api/chat",
                        json={"message": q, "district": "서울 전체"}, timeout=180)
    if res.headers.get("Content-Type", "").startswith("application/json"):
        data = res.json()
        rows.append({"질문": q, "응답 코드": res.status_code, "자치구": data.get("district"),
                     "답변 방식": data.get("source"), "답변": data.get("answer")})
    else:
        rows.append({"질문": q, "응답 코드": res.status_code, "자치구": None,
                     "답변 방식": "json 아님", "답변": res.text[:150]})

display(pd.DataFrame(rows).style.set_properties(subset=["답변"], **{"white-space": "pre-wrap"}))

## 4. 동 이름 → 자치구 사전 만들기 (처음 한 번)
119안전센터 관할구역 글의 "○○구중 : △△동, …" 부분을 읽어 `dong_to_gu.json`으로 저장

- "역삼 1ㆍ2동", "을지로3~7가" → 숫자·기호를 지운 이름(역삼동, 을지로가)으로 저장
- 같은 동 이름이 여러 구에 있으면 구 목록으로 저장 (예: 신사동 → 강남구·관악구·은평구)
- 저장 위치 : chatbot_exaone.py 옆 (서버 재시작 후 적용)

In [ ]:
import os
import re
import json
import importlib
import chatbot_exaone

CSV_PATH = r"C:\ai\source\09_Project\cheolhyeon_jo\data\서울소방서 119안전센터 현황.csv"

df = pd.read_csv(CSV_PATH, encoding="cp949")
gu_pattern = re.compile("(" + "|".join(sorted(chatbot_exaone.GU_LIST, key=len, reverse=True)) + r")\s*중\s*:")

dong_map = {}
for text in df["관할구역"]:
    hits = list(gu_pattern.finditer(text))
    for i, hit in enumerate(hits):
        end = hits[i + 1].start() if i + 1 < len(hits) else None
        body = re.sub(r"([가-힣])\s+(\d)", r"\1\2", text[hit.end():end])  # "역삼 1동" → "역삼1동"
        for token in re.findall(r"[가-힣]+\d*(?:[ㆍ~?]\s*\d+)*(?:동|가)", body):
            name = re.sub(r"[\dㆍ~?\s]+", "", token)
            if len(name) >= 2:
                dong_map.setdefault(name, set()).add(hit.group(1))

for wrong in ["동동", "신동", "극동"]:  # 아파트 이름 등에서 잘못 잡힌 이름
    dong_map.pop(wrong, None)

dong_map = {k: sorted(v) for k, v in sorted(dong_map.items())}
with open(os.path.join(CHATBOT_DIR, "dong_to_gu.json"), "w", encoding="utf-8") as f:
    json.dump(dong_map, f, ensure_ascii=False, indent=0)

print("동 이름 수 :", len(dong_map))
display(pd.DataFrame([(k, v) for k, v in dong_map.items() if len(v) > 1], columns=["동", "자치구 후보"]))

## 5. 질문 분류 확인
- 주소 → 가장 가까운 응급실 (가용 병상 1개 이상)
- 구·동 → 이송 수요 · 생활인구 예측
- 장소 없이 "근처 응급실" → 브라우저 현재 위치

In [ ]:
importlib.reload(chatbot_exaone)  # 새로 만든 사전 다시 읽기

def route(q):
    gu, dong, cands = chatbot_exaone.find_place(q)
    address = chatbot_exaone.find_address(q)
    near = bool(chatbot_exaone.NEAR_WORDS.search(q))
    if address:
        how = "가까운 응급실 (주소)"
    elif not gu and len(cands) > 1:
        how = "구 다시 묻기"
    elif gu and near:
        how = "가까운 응급실 (동네 중심)"
    elif gu:
        how = "예측"
    elif near:
        how = "가까운 응급실 (현재 위치)"
    else:
        how = "서울 전체 예측 또는 안내"
    return q, address, gu, dong, how

places = ["테헤란로 152 응급실", "서울 강남구 테헤란로 152", "역삼동 123-4", "역삼1동 이송 예측",
          "을지로3가", "신사동 예측", "강남구 신사동", "역삼동 근처 응급실", "근처 응급실"]
display(pd.DataFrame([route(q) for q in places], columns=["질문", "주소", "자치구", "동", "처리"]))